# Thulla DMC — Colab T4 training

Minimal DouZero-style self-play for a decent 4-player Thulla bot.

Learns **card plays** and **ASK/PASS** on the take phase (victims always give).

Obs include known holdings (thulla/take) and the **free unknown** card pool; heads-up uses deduced opponent hands.

**Setup:** Runtime → Change runtime type → **T4 GPU** (not CPU).

- Learner runs on the **T4 GPU**
- Self-play actors run on **CPU** (feeds the GPU) — default 6 actors
- Checkpoints every ~3 min to Drive

**Note:** If you changed obs/encoding recently, start a **fresh** checkpoint folder or delete old `model.tar`.

## 1. Install deps

In [ ]:
%pip install -q "numpy>=1.24" "torch>=2.0"

## 2. Get thulla-ai code

Either clone your repo, or upload a zip of `thulla-ai` to Drive and set `REPO_DIR` below.

In [ ]:
import os
import sys

import torch

# --- edit these ---
REPO_URL = ""  # e.g. "https://github.com/YOU/thulla-ai.git" or leave blank if uploading
REPO_DIR = "/content/thulla-ai"
DRIVE_CKPT = "/content/drive/MyDrive/thulla_dmc_ckpts"
# ------------------

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_CKPT, exist_ok=True)

if REPO_URL and not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
elif not os.path.isdir(REPO_DIR):
    raise SystemExit(
        f"Missing {REPO_DIR}. Set REPO_URL or upload thulla-ai there (must contain thulla/ and thulla_dmc/)."
    )

sys.path.insert(0, REPO_DIR)

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU. Runtime → Change runtime type → Hardware accelerator → T4 GPU, then re-run."
    )

print("REPO_DIR:", REPO_DIR)
print("checkpoints:", DRIVE_CKPT)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


## 3. Train on T4 (auto-resume if checkpoint exists)

Defaults tuned for **Colab T4**: GPU learner, 6 CPU actors, batch 512, checkpoint every 3 minutes. Stop anytime — weights are on Drive.


In [ ]:
import os
from thulla_dmc.arguments import parse_args
from thulla_dmc.train import train

ckpt_dir = os.path.join(DRIVE_CKPT, "thulla_dmc")
has_ckpt = os.path.exists(os.path.join(ckpt_dir, "model.tar"))

# Colab T4 preset — bump --num_actors to 8 if the GPU looks idle
argv = [
    "--savedir", DRIVE_CKPT,
    "--xpid", "thulla_dmc",
    "--training_device", "0",   # T4 GPU
    "--require_gpu",
    "--num_actors", "6",        # CPU self-play workers feeding the GPU
    "--batch_size", "512",
    "--save_interval", "3",     # minutes
    "--total_episodes", "30000",
    "--exp_epsilon", "0.05",
    "--log_interval", "25",
]
if has_ckpt:
    argv.append("--load_model")
    print("Resuming from", ckpt_dir)
else:
    print("Starting fresh →", ckpt_dir)

flags = parse_args(argv)
train(flags)


## 4. Evaluate vs RandomPlayer

Primary metric: **P(not last)** (random ≈ 0.75 for a single seat among 4).

In [ ]:
from thulla_dmc.evaluate import evaluate

ckpt = f"{DRIVE_CKPT}/thulla_dmc/model.tar"
evaluate(ckpt, num_games=400, device="0")